# 1. IMAGE SEGMENTATION

## 1.1 THRESHOLD-BASED SEGMENTATION

El objetivo de esta sección es detectar defectos en una radiografía de una soldadura. Sabemos que los defectos dejan pasar más los rayos X, por lo que se ven como las regiones más brillantes del interior.Para lograr aislar estos defectos, aplicaremos una técnica llamada Umbralización (Thresholding).
La regla matemática básica es:Si el valor del píxel $f(x,y) > T$ (siendo $T$ nuestro umbral), el nuevo píxel será Blanco (1 o 255).Si el valor del píxel $f(x,y) \le T$, el nuevo píxel será Negro (0).¿Cómo elegimos $T$?Umbral de Otsu: El laboratorio nos pide usar primero la función threshold_otsu. Este es un algoritmo que analiza el histograma de la imagen (la distribución de grises) y calcula automáticamente el valor $T$ óptimo que separa la imagen en dos clases equilibradas (fondo y objeto), minimizando la varianza estadística dentro de cada clase.Umbral Manual: Otsu funciona muy bien cuando hay dos montañas claras (bimodal) en el histograma. Sin embargo, en una radiografía, los defectos brillantes son muy pequeños en área comparados con el resto de la soldadura. Esto engaña a Otsu. Por eso, el laboratorio nos pide luego ver el histograma para elegir a ojo un $T$ más alto que realmente aísle la "cola" brillante derecha del gráfico.Filtro Gaussiano: Las imágenes médicas suelen tener ruido granulado. Si aplicamos el umbral directamente, obtendremos muchos "puntos" blancos falsos. Aplicaremos un filtro paso bajo Gaussiano, que funciona difuminando la imagen (promediando cada píxel con sus vecinos en forma de campana) para homogeneizar las manchas antes de aplicar el umbral.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color, filters, exposure
from skimage.morphology import rectangle, diamond, binary_opening, binary_dilation
from sklearn.cluster import KMeans
from scipy.ndimage import uniform_filter, generic_filter

In [ ]:
# 1. Leer la imagen y asegurarnos de que esté en escala de grises
img1 = io.imread('Im1.jpg', as_gray=True)

# 2. Calcular el umbral de Otsu y aplicar la máscara binaria
t_otsu = filters.threshold_otsu(img1)
mask_otsu = img1 > t_otsu

# 3. Visualizar la imagen original, la máscara de Otsu y el histograma
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Mostrar imagen original
axes[0].imshow(img1, cmap='gray')
axes[0].set_title('Original - Im1')
axes[0].axis('off')

# Mostrar segmentación de Otsu
axes[1].imshow(mask_otsu, cmap='gray')
axes[1].set_title(f'Segmentación Otsu (T={t_otsu:.2f})')
axes[1].axis('off')

# Mostrar el histograma
# img1.ravel() aplana la matriz 2D a 1D para calcular el histograma
axes[2].hist(img1.ravel(), bins=256, color='gray', alpha=0.7)
axes[2].axvline(t_otsu, color='r', linestyle='dashed', linewidth=2, label=f'Otsu T={t_otsu:.2f}')
axes[2].set_title('Histograma de niveles de gris')
axes[2].legend()

plt.tight_layout()
plt.show()